<a href="https://colab.research.google.com/github/higorfigueredo1996/projeto-agente-de-bulas-farmaceuticas/blob/main/Projeto_2_Bulas_Farmac%C3%AAuticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-classic langchain-groq langchain-huggingface chromadb pypdf sentence-transformers

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

print("Importações e chave carregadas com sucesso.")

Importações e chave carregadas com sucesso.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Ajuste o caminho da pasta conforme você criou no seu Drive
PASTA = "/content/drive/MyDrive/Projetos Rag/"

caminhos_bulas = [
    PASTA + "bula dipirona.pdf",
    PASTA + "bula paracetamol.pdf",
]

documentos = []

for caminho in caminhos_bulas:
    loader = PyPDFLoader(caminho)
    docs = loader.load()
    # Adiciona o nome do medicamento como metadado, extraído do nome do arquivo
    nome_medicamento = caminho.split("/")[-1].replace(".pdf", "")
    for doc in docs:
        doc.metadata["medicamento"] = nome_medicamento
    documentos.extend(docs)

print(f"Total de páginas carregadas: {len(documentos)}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)
chunks = text_splitter.split_documents(documentos)
print(f"Total de chunks gerados: {len(chunks)}")

Total de páginas carregadas: 4
Total de chunks gerados: 70


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/"))

['Ideias de E-books.gsheet', 'Tamanho imagem destacada.gdoc', 'COMO AGENDAR SEUS POSTS NO FACEBOOK E INSTAGRAM DE GRAÇA.gdoc', 'COMO VENDER COMO AFILIADO SEM APARECER.gdoc', 'Trafego orgânico e trafego pago.gdoc', 'HOTMART.gdoc', 'GUIA COM OS PRINCIPAIS TAMANHOS DE IMAGENS PARA REDES SOCIAIS [2020].gdoc', 'MARKETING DE RELACIONAMENTO.gdoc', 'Os Melhores Geradores de Nome de Domínio.gdoc', 'E-book – Como trabalhar em casa (título temporário).gdoc', 'Tema Blog.gdoc', 'Melhores Sites Para Ganhar Dinheiro.gdoc', 'Ferramentas de Marketing de Conteúdo.gdoc', 'Melhores Ferramentas de Roteiros 2020.gdoc', 'Plataformas Digitais: O Que São e Como Usá-las Para Impulsionar seu Negócio.gdoc', 'Visualização de boleto (1).pdf', 'Visualização de boleto.pdf', 'Melhores Plataformas de Afiliados do Brasil e do Mundo.gdoc', 'O Que é Freelancer.gdoc', 'Como Usar as Mídias Sociais Para Marketing De Afiliados.gdoc', 'Ubersuggest.gdoc', 'Marketing Boca a Boca.gdoc', 'Ideias de video Instagram.g

In [ ]:
for chunk in chunks:
    texto = chunk.page_content.lower()

    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"
    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"
    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"
    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"
    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"
    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"
    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"
    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"
    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"
    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"
    else:
        chunk.metadata["categoria"] = "geral"

print("Categorias atribuídas a todos os chunks.")

Categorias atribuídas a todos os chunks.


In [ ]:
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Chunk Aleatório {i} ---")
    print(f"Medicamento: {chunk.metadata.get('medicamento')}")
    print(f"Categoria: {chunk.metadata.get('categoria')}")
    print("\nConteúdo (início):")
    print(chunk.page_content[:300])


--- Chunk Aleatório 1 ---
Medicamento: bula paracetamol
Categoria: como_funciona

Conteúdo (início):
dicamento): urticária, coceira e vermelhidão no 
corpo, reações alérgicas a este medicamento e 
aumento das transaminases.
Informe ao seu médico, cirurgião-dentista ou 
farmacêutico o aparecimento de reações in-
desejáveis pelo uso do medicamento. Informe 
também à empresa através do seu serviço de 

--- Chunk Aleatório 2 ---
Medicamento: bula dipirona
Categoria: contraindicacao

Conteúdo (início):
sintomas relacionados a essas reações cutâneas graves 
descritas na seção “4. O que devo saber antes de usar 
este medicamento?”.
Se você desenvolver alguns desses sinais ou sintomas, 
erupções cutâneas muitas vezes com bolhas ou lesões 
da mucosa, o tratamento deve ser interrompido 
imediatamente e


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_bulas",
)
print("Vector store criada e populada.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store criada e populada.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

system_prompt = (
    "Use o contexto abaixo, extraído das bulas de medicamentos, para responder à pergunta do paciente. "
    "Se a informação não estiver no contexto, diga que não sabe e recomende procurar um médico ou farmacêutico. "
    "Responda de forma clara, direta e em português.\n\n"
    "Contexto: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

print("Cadeia RAG montada com sucesso.")

Cadeia RAG montada com sucesso.


In [ ]:
perguntas = [
    "Quais são as contraindicações da dipirona?",
    "Qual é a posologia recomendada do paracetamol para adultos?",
]

for pergunta in perguntas:
    resposta = chain.invoke({"input": pergunta})
    print(f"Pergunta: {pergunta}")
    print(f"\nResposta do Agente:\n{resposta['answer']}")
    print("\nTrechos utilizados como contexto:\n")
    for i, doc in enumerate(resposta["context"], start=1):
        print(f"--- Trecho {i} ---")
        print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
        print(f"Categoria: {doc.metadata.get('categoria', 'N/A')}")
        print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("\n" + "=" * 50)

Pergunta: Quais são as contraindicações da dipirona?

Resposta do Agente:
De acordo com o contexto fornecido, as contraindicações da dipirona incluem:

1. Pacientes que já tomaram algum medicamento contendo dipirona e tiveram problemas de fígado.
2. Pacientes com lesão hepática (lesão do fígado) durante o tratamento com dipirona, sem causa determinada.
3. Amamentação: a dipirona é eliminada no leite materno, portanto, a amamentação deve ser evitada durante e por até 48 horas após o uso de dipirona.

Além disso, é recomendada supervisão médica quando se administra dipirona a crianças pequenas e pacientes idosos, pois as funções do fígado e dos rins podem estar prejudicadas.

É importante notar que a dipirona também pode causar reações anafilactoides em pacientes que já tiveram reações semelhantes a outros analgésicos não narcóticos. Portanto, é fundamental consultar um médico antes de tomar dipirona, especialmente se você tiver alguma condição médica pré-existente ou estiver tomando out

In [ ]:
retriever_filtrado = vectorstore.as_retriever(
    search_kwargs={"k": 4, "filter": {"categoria": "contraindicacao"}}
)

question_answer_chain_filtrado = create_stuff_documents_chain(llm, prompt)
chain_filtrada = create_retrieval_chain(retriever_filtrado, question_answer_chain_filtrado)

pergunta = "Quais são as contraindicações da dipirona?"
resposta = chain_filtrada.invoke({"input": pergunta})

print(f"Pergunta: {pergunta}")
print(f"\nResposta do Agente (com filtro):\n{resposta['answer']}")
print("\nTrechos utilizados como contexto:\n")
for i, doc in enumerate(resposta["context"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Categoria: {doc.metadata.get('categoria', 'N/A')}")

Pergunta: Quais são as contraindicações da dipirona?

Resposta do Agente (com filtro):
Não sei, pois as contraindicações da dipirona não estão explicitamente mencionadas no contexto fornecido. Recomendo procurar um médico ou farmacêutico para obter informações precisas e atualizadas sobre as contraindicações da dipirona. Além disso, é importante consultar a bula do medicamento ou outras fontes confiáveis para obter informações detalhadas sobre seu uso seguro e adequado.

Trechos utilizados como contexto:

--- Trecho 1 ---
Medicamento: bula dipirona
Categoria: contraindicacao
